# 09v VeRi-776 Evaluation: TransReID ViT-B/16 (CLIP) + SIE v7 Sweep

Goal: push R1 past 98% on VeRi-776 with 256x256 + SIE features, rerank and AQE sweeps, an AQE x rerank grid, and targeted 10-crop TTA checks.

In [ ]:
import sys, os, subprocess
print("Python:", sys.version)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm==1.0.11", "loguru"], check=True)
print("Deps installed")

In [ ]:
# Discover mounted input directories (paths can vary)
import os
for root in sorted(os.listdir("/kaggle/input")):
    full = os.path.join("/kaggle/input", root)
    print(f"{full}/")
    try:
        for sub in sorted(os.listdir(full))[:8]:
            print(f"  -> {sub}")
    except Exception as e:
        print(f"  ERR: {e}")

In [ ]:
# Resolve VeRi root and Checkpoint A via deep recursive search
import os
from pathlib import Path

VERI_ROOT = None
WEIGHTS_VERI = None

for root, dirs, files in os.walk('/kaggle/input'):
    p = Path(root)
    if VERI_ROOT is None and (p / 'image_query').is_dir() and (p / 'image_test').is_dir():
        VERI_ROOT = p
    if WEIGHTS_VERI is None and 'vehicle_transreid_vit_base_veri776.pth' in files:
        WEIGHTS_VERI = str(p / 'vehicle_transreid_vit_base_veri776.pth')

assert VERI_ROOT is not None, 'VeRi-776 not found'
assert WEIGHTS_VERI is not None, 'VeRi checkpoint not found'
print('VeRi root  :', VERI_ROOT)
print('VeRi ckpt  :', WEIGHTS_VERI)

In [ ]:
# Clone the gp repo to access src/training/evaluate_reid.py and src/stage2_features/transreid_model.py
import subprocess, os, sys
if not os.path.isdir("/kaggle/working/gp"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/MRKDaGods/gp.git", "/kaggle/working/gp"], check=True)
if "/kaggle/working/gp" not in sys.path:
    sys.path.insert(0, "/kaggle/working/gp")
print(os.listdir("/kaggle/working/gp/src")[:10])

In [ ]:
# Parse VeRi-776 query / gallery and reproduce the training-time camera mapping
import re
from pathlib import Path

CAMERA_PATTERN = re.compile(r"^(?P<pid>-?\d+)_c(?P<camid>\d+)")
NUM_VERI_CAMERAS = 20

def parse_veri_record(img_path):
    match = CAMERA_PATTERN.match(img_path.stem)
    if match is None:
        raise RuntimeError(f"Unexpected VeRi filename: {img_path.name}")
    pid = int(match.group("pid"))
    parsed_camid = int(match.group("camid"))
    if not 1 <= parsed_camid <= NUM_VERI_CAMERAS:
        raise RuntimeError(f"Camera ID out of range for {img_path.name}: c{parsed_camid:03d}")
    sie_index = parsed_camid - 1
    return {
        "path": str(img_path),
        "pid": pid,
        "parsed_camid": parsed_camid,
        "sie_index": sie_index,
    }

def parse_split(split_dir):
    items = []
    pid_set = set()
    for img_path in sorted(Path(split_dir).glob("*.jpg")):
        record = parse_veri_record(img_path)
        if record["pid"] == -1:
            continue
        pid_set.add(record["pid"])
        items.append(record)
    return items, len(pid_set)

query, n_q = parse_split(VERI_ROOT / "image_query")
gallery, n_g = parse_split(VERI_ROOT / "image_test")
sample_cam_tuples = [
    (item["path"], item["parsed_camid"], item["sie_index"])
    for item in (query + gallery)[:5]
]

print(f"Query:   {len(query):,} images, {n_q} IDs")
print(f"Gallery: {len(gallery):,} images, {n_g} IDs")
print("First 5 (path, parsed_camid, sie_index) tuples:")
for row in sample_cam_tuples:
    print(" ", row)

In [ ]:
# DataLoader factory
import torch
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
from PIL import Image

CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD = [0.26862954, 0.26130258, 0.27577711]

class VeRiDataset(Dataset):
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        item = self.items[index]
        image = Image.open(item["path"]).convert("RGB")
        return self.transform(image), item["pid"], item["sie_index"], item["path"]

def build_transform(img_size):
    return T.Compose([
        T.Resize(img_size, interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(CLIP_MEAN, CLIP_STD),
    ])

def batch_size_for_img_size(img_size):
    longest = max(img_size)
    if longest >= 384:
        return 32
    if longest >= 320:
        return 48
    return 64

def build_loader(items, img_size):
    batch_size = batch_size_for_img_size(img_size)
    loader = DataLoader(
        VeRiDataset(items, build_transform(img_size)),
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
    )
    return loader, batch_size

print("DataLoader factory ready")

In [ ]:
from src.stage2_features.transreid_model import build_transreid
from src.training.evaluate_reid import compute_distance_matrix, eval_market1501
import gc
import json
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from PIL import Image
from torch.utils.data import DataLoader, Dataset

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
IMG_SIZE = (224, 224)
MULTISCALE_SIZES = [(224, 224), (256, 256)]
CONCAT_PATCH_GEM_P = 3.0
RESULTS_PATH = "/kaggle/working/veri776_eval_results_v10.json"
AQE_K_VALUES = [1, 2, 3]
AQE_ITER2_K = 3
CROSS_AQE_K_VALUES = list(AQE_K_VALUES)
MAX_TIME_PER_RERANK = 1500
SINGLE_FLIP_BATCH_SIZE = 64
ENABLE_TEN_CROP = False
TEN_CROP_BATCH_SIZE = 32
TEN_CROP_RESIZE = (288, 288)
SIE_NUM_CAMERAS = 20
EVAL_NO_SIE = False
RERANK_CONFIGS = [
    {"label": "k1=22,k2=6,lambda=0.2",  "k1": 22, "k2": 6,  "lambda_value": 0.2},
    {"label": "k1=22,k2=7,lambda=0.2",  "k1": 22, "k2": 7,  "lambda_value": 0.2},
    {"label": "k1=24,k2=7,lambda=0.2",  "k1": 24, "k2": 7,  "lambda_value": 0.2},
    {"label": "k1=24,k2=8,lambda=0.2",  "k1": 24, "k2": 8,  "lambda_value": 0.2},
    {"label": "k1=25,k2=7,lambda=0.2",  "k1": 25, "k2": 7,  "lambda_value": 0.2},
    {"label": "k1=25,k2=8,lambda=0.2",  "k1": 25, "k2": 8,  "lambda_value": 0.2},
    {"label": "k1=25,k2=8,lambda=0.18", "k1": 25, "k2": 8,  "lambda_value": 0.18},
    {"label": "k1=25,k2=8,lambda=0.22", "k1": 25, "k2": 8,  "lambda_value": 0.22},
    {"label": "k1=26,k2=8,lambda=0.2",  "k1": 26, "k2": 8,  "lambda_value": 0.2},
    {"label": "k1=28,k2=8,lambda=0.2",  "k1": 28, "k2": 8,  "lambda_value": 0.2},
    {"label": "k1=28,k2=9,lambda=0.2",  "k1": 28, "k2": 9,  "lambda_value": 0.2},
    {"label": "k1=30,k2=10,lambda=0.2", "k1": 30, "k2": 10, "lambda_value": 0.2},
    {"label": "k1=30,k2=10,lambda=0.15","k1": 30, "k2": 10, "lambda_value": 0.15},
    {"label": "k1=50,k2=15,lambda=0.2", "k1": 50, "k2": 15, "lambda_value": 0.2},
    {"label": "k1=80,k2=15,lambda=0.2", "k1": 80, "k2": 15, "lambda_value": 0.2},
]
EXACT_08_BASELINE_RERANK = {"label": "k1=20,k2=6,lambda=0.3", "k1": 20, "k2": 6, "lambda_value": 0.3}
GUARANTEED_AQE_RERANK_PAIRS = []

print("Device:", DEVICE)
print("Results path:", RESULTS_PATH)
print("AQE sweep:", AQE_K_VALUES)
print("AQE x rerank cross sweep:", CROSS_AQE_K_VALUES)
print("Multi-scale sizes:", MULTISCALE_SIZES)
print("Concat-patch GeM p:", CONCAT_PATCH_GEM_P)
print("Rerank guard (sec):", MAX_TIME_PER_RERANK)
print("Ten-crop enabled:", ENABLE_TEN_CROP)
print("No-SIE eval enabled:", EVAL_NO_SIE)


def load_model(weights_path):
    model = build_transreid(
        num_classes=1,
        num_cameras=SIE_NUM_CAMERAS,
        embed_dim=768,
        vit_model="vit_base_patch16_clip_224.openai",
        pretrained=False,
        weights_path=weights_path,
        img_size=IMG_SIZE,
    )
    model._concat_patch = False
    model._gem_p = CONCAT_PATCH_GEM_P
    return model.to(DEVICE).eval()


class PathDataset(Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        item = self.items[index]
        if not isinstance(item, dict):
            raise TypeError(f"Expected VeRi item dict, got {type(item).__name__}: {item!r}")
        return item["path"], int(item["pid"]), int(item["sie_index"])


def make_loader(items, batch_size):
    return DataLoader(
        PathDataset(items),
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
    )

In [ ]:
def to_metric_dict(mAP, cmc):
    ranks = list(cmc)
    return {
        "mAP": float(mAP),
        "R1": float(ranks[min(0, len(ranks) - 1)]),
        "R5": float(ranks[min(4, len(ranks) - 1)]),
        "R10": float(ranks[min(9, len(ranks) - 1)]),
    }


def metric_sort_key(metrics):
    return (metrics["R1"], metrics["mAP"], metrics["R5"], metrics["R10"])


def print_metrics(label, metrics, duration_sec=None):
    suffix = "" if duration_sec is None else f"  ({duration_sec:.1f}s)"
    print(
        f"  [{label}] mAP={metrics['mAP'] * 100:.4f}%  "
        f"R1={metrics['R1'] * 100:.4f}%  "
        f"R5={metrics['R5'] * 100:.2f}%  "
        f"R10={metrics['R10'] * 100:.2f}%{suffix}"
    )


def resize_and_normalize(image, size):
    resized = TF.resize(image, size, interpolation=T.InterpolationMode.BICUBIC)
    tensor = TF.to_tensor(resized)
    return TF.normalize(tensor, mean=CLIP_MEAN, std=CLIP_STD)


def build_single_flip_views(image, img_size=IMG_SIZE):
    base = resize_and_normalize(image, img_size)
    return [base, torch.flip(base, dims=[2])]


def build_ten_crop_views(image):
    resized = TF.resize(image, TEN_CROP_RESIZE, interpolation=T.InterpolationMode.BICUBIC)
    crops = TF.five_crop(resized, IMG_SIZE)
    views = []
    for crop in crops:
        normalized = TF.normalize(TF.to_tensor(crop), mean=CLIP_MEAN, std=CLIP_STD)
        views.append(normalized)
        views.append(torch.flip(normalized, dims=[2]))
    return views


def build_view_batches(paths, mode, img_size=IMG_SIZE):
    num_views = 2 if mode in {"single_flip", "single_flip_no_sie", "concat_patch_flip"} else 10
    view_batches = [[] for _ in range(num_views)]
    for path in paths:
        with Image.open(path) as image_handle:
            image = image_handle.convert("RGB")
            views = (
                build_single_flip_views(image, img_size)
                if mode in {"single_flip", "single_flip_no_sie", "concat_patch_flip"}
                else build_ten_crop_views(image)
            )
        for view_index, view in enumerate(views):
            view_batches[view_index].append(view)
    return [torch.stack(batch, dim=0) for batch in view_batches]


def make_dataloader_at_size(items, batch_size, size):
    loader, actual_batch_size = build_loader(items, size)
    if int(actual_batch_size) != int(batch_size):
        print(
            f"  Adjusted batch size for size={size}: requested={batch_size}, actual={actual_batch_size}"
        )
    return loader


def set_concat_patch_mode(model, enabled):
    model._concat_patch = bool(enabled)
    model._gem_p = CONCAT_PATCH_GEM_P


@torch.no_grad()
def extract_features_from_tensor_loader(model, dataloader, disable_sie=False):
    model.eval()
    all_features = []
    all_pids = []
    all_camids = []

    for imgs, pids, camids, _ in dataloader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        real_cam_tensor = camids.to(DEVICE, non_blocking=True).long()
        cam_tensor = torch.zeros_like(real_cam_tensor) if disable_sie else real_cam_tensor

        features = model(imgs, cam_ids=cam_tensor)
        if isinstance(features, (tuple, list)):
            features = features[-1]
        features = F.normalize(features.float(), p=2, dim=1)

        imgs_flip = torch.flip(imgs, dims=[3])
        features_flip = model(imgs_flip, cam_ids=cam_tensor)
        if isinstance(features_flip, (tuple, list)):
            features_flip = features_flip[-1]
        features_flip = F.normalize(features_flip.float(), p=2, dim=1)

        batch_features = F.normalize((features + features_flip) / 2.0, p=2, dim=1)
        all_features.append(batch_features.cpu().numpy())
        all_pids.append(pids.numpy())
        all_camids.append(camids.numpy())

    return (
        np.concatenate(all_features, axis=0),
        np.concatenate(all_pids, axis=0),
        np.concatenate(all_camids, axis=0),
    )


@torch.no_grad()
def extract_features(model, dataloader, mode):
    model.eval()
    if mode == "multi_scale_flip":
        scale_features = []
        final_pids = None
        final_camids = None
        items = dataloader.dataset.items
        for size in MULTISCALE_SIZES:
            scale_loader = make_dataloader_at_size(items, batch_size=SINGLE_FLIP_BATCH_SIZE, size=size)
            features, pids, camids = extract_features_from_tensor_loader(model, scale_loader, disable_sie=False)
            scale_features.append(features.astype(np.float32, copy=False))
            if final_pids is None:
                final_pids = pids
                final_camids = camids
        merged = np.zeros_like(scale_features[0], dtype=np.float32)
        for features in scale_features:
            merged += features
        merged /= np.linalg.norm(merged, axis=1, keepdims=True) + 1e-12
        return merged, final_pids, final_camids

    if mode == "concat_patch_flip":
        tensor_loader = make_dataloader_at_size(
            dataloader.dataset.items,
            batch_size=SINGLE_FLIP_BATCH_SIZE,
            size=IMG_SIZE,
        )
        set_concat_patch_mode(model, True)
        try:
            return extract_features_from_tensor_loader(model, tensor_loader, disable_sie=False)
        finally:
            set_concat_patch_mode(model, False)

    all_features = []
    all_pids = []
    all_camids = []
    disable_sie = mode == "single_flip_no_sie"

    for paths, pids, camids in dataloader:
        real_cam_tensor = camids.to(DEVICE, non_blocking=True).long()
        cam_tensor = torch.zeros_like(real_cam_tensor) if disable_sie else real_cam_tensor
        view_batches = build_view_batches(list(paths), mode)
        per_view_features = []
        for view_batch in view_batches:
            view_batch = view_batch.to(DEVICE, non_blocking=True)
            features = model(view_batch, cam_ids=cam_tensor)
            if isinstance(features, (tuple, list)):
                features = features[-1]
            per_view_features.append(F.normalize(features.float(), p=2, dim=1).cpu())
        batch_features = F.normalize(torch.stack(per_view_features, dim=0).mean(dim=0), p=2, dim=1)
        all_features.append(batch_features.numpy())
        all_pids.append(pids.numpy())
        all_camids.append(camids.numpy())

    return (
        np.concatenate(all_features, axis=0),
        np.concatenate(all_pids, axis=0),
        np.concatenate(all_camids, axis=0),
    )


def extract_feature_bundle(model, mode, query_items, gallery_items):
    batch_size = (
        SINGLE_FLIP_BATCH_SIZE
        if mode in {"single_flip", "single_flip_no_sie", "concat_patch_flip", "multi_scale_flip"}
        else TEN_CROP_BATCH_SIZE
    )
    q_loader_local = make_loader(query_items, batch_size=batch_size)
    g_loader_local = make_loader(gallery_items, batch_size=batch_size)
    started = time.time()
    q_features, q_pids, q_camids = extract_features(model, q_loader_local, mode)
    g_features, g_pids, g_camids = extract_features(model, g_loader_local, mode)
    elapsed = time.time() - started
    print(f"  Extracted {mode} features in {elapsed:.1f}s -- q={q_features.shape}, g={g_features.shape}")
    if mode == "multi_scale_flip":
        view_count = 2 * len(MULTISCALE_SIZES)
    elif mode in {"single_flip", "single_flip_no_sie", "concat_patch_flip"}:
        view_count = 2
    else:
        view_count = 10
    return {
        "mode": mode,
        "sie_disabled_at_eval": bool(mode == "single_flip_no_sie"),
        "view_count": view_count,
        "feature_extraction_sec": float(elapsed),
        "q_features": q_features,
        "g_features": g_features,
        "q_pids": q_pids,
        "g_pids": g_pids,
        "q_camids": q_camids,
        "g_camids": g_camids,
    }


def evaluate_cosine_features(q_features, g_features, q_pids, g_pids, q_camids, g_camids):
    distmat = compute_distance_matrix(q_features, g_features, metric="cosine")
    mAP, cmc = eval_market1501(distmat, q_pids, g_pids, q_camids, g_camids)
    return to_metric_dict(mAP, cmc)


def average_query_expansion(features, k, iterations=1):
    current = features.astype(np.float32, copy=True)
    if k <= 1:
        return current
    for _ in range(iterations):
        sim = current @ current.T
        topk = min(k, sim.shape[1])
        kth = max(topk - 1, 0)
        topk_idx = np.argpartition(-sim, kth=kth, axis=1)[:, :topk]
        expanded = np.zeros_like(current)
        for index in range(current.shape[0]):
            expanded[index] = current[topk_idx[index]].mean(axis=0)
        norms = np.linalg.norm(expanded, axis=1, keepdims=True) + 1e-12
        current = expanded / norms
    return current


@torch.no_grad()
def build_rerank_state(all_features, max_k1):
    features = torch.as_tensor(all_features, dtype=torch.float32, device=DEVICE)
    features = F.normalize(features, p=2, dim=1)
    similarity = torch.matmul(features, features.T)
    original_dist = (2.0 - 2.0 * similarity).clamp_min_(0).cpu().numpy().astype(np.float32)
    initial_rank = torch.topk(
        similarity,
        k=min(max_k1 + 1, similarity.shape[1]),
        dim=1,
        largest=True,
        sorted=True,
    ).indices.cpu().numpy().astype(np.int32)
    del features, similarity
    if DEVICE.startswith("cuda"):
        torch.cuda.empty_cache()
    return original_dist, initial_rank


def compute_reranking_torch(original_dist, initial_rank, query_num, k1=20, k2=6, lambda_value=0.3):
    all_num = original_dist.shape[0]
    V = np.zeros((all_num, all_num), dtype=np.float16)
    half_k1 = int(np.round(k1 / 2.0))

    for index in range(all_num):
        forward = initial_rank[index, :k1 + 1]
        backward = initial_rank[forward, :k1 + 1]
        reciprocal = forward[np.any(backward == index, axis=1)]
        reciprocal_expansion = reciprocal.copy()

        for candidate in reciprocal:
            candidate_forward = initial_rank[candidate, :half_k1 + 1]
            candidate_backward = initial_rank[candidate_forward, :half_k1 + 1]
            candidate_reciprocal = candidate_forward[np.any(candidate_backward == candidate, axis=1)]
            if candidate_reciprocal.size == 0:
                continue
            overlap = np.intersect1d(candidate_reciprocal, reciprocal)
            if overlap.size > (2.0 / 3.0) * candidate_reciprocal.size:
                reciprocal_expansion = np.concatenate((reciprocal_expansion, candidate_reciprocal))

        reciprocal_expansion = np.unique(reciprocal_expansion)
        weights = np.exp(-original_dist[index, reciprocal_expansion]).astype(np.float32)
        V[index, reciprocal_expansion] = (weights / (weights.sum() + 1e-12)).astype(np.float16)

    if k2 > 1:
        V_qe = np.zeros_like(V, dtype=np.float16)
        for index in range(all_num):
            V_qe[index] = V[initial_rank[index, :k2]].mean(axis=0)
        V = V_qe

    inv_index = [np.flatnonzero(V[:, column]) for column in range(all_num)]
    jaccard_dist = np.zeros((query_num, all_num), dtype=np.float32)

    for index in range(query_num):
        temp_min = np.zeros(all_num, dtype=np.float32)
        non_zero = np.flatnonzero(V[index])
        for nz in non_zero:
            related = inv_index[nz]
            temp_min[related] += np.minimum(np.float32(V[index, nz]), V[related, nz].astype(np.float32))
        jaccard_dist[index] = 1.0 - temp_min / (2.0 - temp_min)

    final_dist = jaccard_dist * (1.0 - lambda_value) + original_dist[:query_num] * lambda_value
    return final_dist[:, query_num:]


def evaluate_rerank_variant(state, q_pids, g_pids, q_camids, g_camids, cfg, variant_label):
    started = time.time()
    original_dist, initial_rank = state
    distmat = compute_reranking_torch(
        original_dist=original_dist,
        initial_rank=initial_rank,
        query_num=len(q_pids),
        k1=cfg["k1"],
        k2=cfg["k2"],
        lambda_value=cfg["lambda_value"],
    )
    mAP, cmc = eval_market1501(distmat, q_pids, g_pids, q_camids, g_camids)
    elapsed = time.time() - started
    metrics = to_metric_dict(mAP, cmc)
    metrics["duration_sec"] = float(elapsed)
    print_metrics(f"{variant_label} {cfg['label']}", metrics, duration_sec=elapsed)
    exceeded = elapsed > MAX_TIME_PER_RERANK
    if exceeded:
        print(
            f"  Rerank guard triggered after {elapsed:.1f}s on {variant_label} {cfg['label']}; "
            "skipping remaining rerank variants."
        )
    return metrics, exceeded


def summarize_records(records, label, limit=5):
    ranked = sorted(records, key=lambda record: metric_sort_key(record["metrics"]), reverse=True)
    print(f"  Top {min(limit, len(ranked))} for {label}:")
    for record in ranked[:limit]:
        print_metrics(record["label"], record["metrics"])
    return ranked


def copy_cfg(cfg):
    return {
        "label": cfg["label"],
        "k1": int(cfg["k1"]),
        "k2": int(cfg["k2"]),
        "lambda_value": float(cfg["lambda_value"]),
    }


def make_candidate(label, feature_bundle, stage, metrics, extra=None):
    candidate = {
        "label": label,
        "feature_bundle": feature_bundle,
        "stage": stage,
        "metrics": metrics,
    }
    if extra is not None:
        candidate.update(extra)
    return candidate

In [ ]:
def run_full_eval(weights_path, label):
    print(f"\n{'=' * 80}\n{label}\n  weights={weights_path}\n{'=' * 80}")
    model = load_model(weights_path)
    total_start = time.time()

    bundle_specs = [
        ("single_flip", False),
        ("concat_patch_flip", False),
    ]
    if EVAL_NO_SIE:
        bundle_specs.append(("single_flip_no_sie", True))

    feature_bundles = {}
    for bundle_name, _ in bundle_specs:
        feature_bundles[bundle_name] = extract_feature_bundle(model, bundle_name, query, gallery)

    results = {
        "metadata": {
            "weights_path": weights_path,
            "weights_name": Path(weights_path).name,
            "img_size": list(IMG_SIZE),
            "multi_scale_sizes": [list(size) for size in MULTISCALE_SIZES],
            "concat_patch_gem_p": float(CONCAT_PATCH_GEM_P),
            "ten_crop_enabled": ENABLE_TEN_CROP,
            "ten_crop_resize": list(TEN_CROP_RESIZE),
            "aqe_k_values": list(AQE_K_VALUES),
            "aqe_iter2_k": AQE_ITER2_K,
            "cross_aqe_k_values": list(CROSS_AQE_K_VALUES),
            "guaranteed_aqe_rerank_pairs": [
                {"aqe_k": int(pair["aqe_k"]), "rerank": copy_cfg(pair["rerank"])}
                for pair in GUARANTEED_AQE_RERANK_PAIRS
            ],
            "max_time_per_rerank_sec": MAX_TIME_PER_RERANK,
            "rerank_configs": [copy_cfg(cfg) for cfg in RERANK_CONFIGS],
            "exact_08_baseline_rerank": copy_cfg(EXACT_08_BASELINE_RERANK),
            "sie_enabled": True,
            "eval_no_sie": EVAL_NO_SIE,
            "sie_num_cameras": SIE_NUM_CAMERAS,
            "single_flip_batch_size": SINGLE_FLIP_BATCH_SIZE,
            "ten_crop_batch_size": TEN_CROP_BATCH_SIZE,
            "evaluated_feature_bundles": [bundle_name for bundle_name, _ in bundle_specs],
        },
        "feature_sets": {
            "ten_crop": {
                "enabled": False,
                "skipped": True,
                "reason": "disabled after underperforming the single-flip baseline",
            } if not ENABLE_TEN_CROP else {},
        },
        "top_candidates": [],
        "best_overall": None,
    }

    all_candidates = []

    for bundle_name, sie_disabled_at_eval in bundle_specs:
        bundle = feature_bundles[bundle_name]
        q_features = bundle["q_features"]
        g_features = bundle["g_features"]
        q_pids = bundle["q_pids"]
        g_pids = bundle["g_pids"]
        q_camids = bundle["q_camids"]
        g_camids = bundle["g_camids"]
        all_features = np.concatenate([q_features, g_features], axis=0)

        bundle_results = {
            "feature_extraction_sec": float(bundle["feature_extraction_sec"]),
            "view_count": int(bundle["view_count"]),
            "query_shape": list(q_features.shape),
            "gallery_shape": list(g_features.shape),
            "sie_disabled_at_eval": bool(sie_disabled_at_eval),
            "baseline": None,
            "exact_08_baseline_rerank": None,
            "rerank_sweep": [],
            "aqe_sweep": [],
            "aqe_iter2": None,
            "aqe_rerank_cross": [],
            "rerank_guard_triggered": False,
        }

        baseline = evaluate_cosine_features(q_features, g_features, q_pids, g_pids, q_camids, g_camids)
        bundle_results["baseline"] = baseline
        print_metrics(f"{bundle_name} baseline", baseline)
        all_candidates.append(make_candidate(f"{bundle_name} baseline", bundle_name, "baseline", baseline))

        max_k1 = max(max(cfg["k1"] for cfg in RERANK_CONFIGS), int(EXACT_08_BASELINE_RERANK["k1"]))
        print(f"  Building rerank state for {bundle_name} base features ...")
        base_state = build_rerank_state(all_features, max_k1=max_k1)

        exact_metrics, exceeded = evaluate_rerank_variant(
            base_state,
            q_pids,
            g_pids,
            q_camids,
            g_camids,
            EXACT_08_BASELINE_RERANK,
            f"{bundle_name} rerank",
        )
        bundle_results["exact_08_baseline_rerank"] = {
            "label": f"{bundle_name} rerank {EXACT_08_BASELINE_RERANK['label']}",
            "config": copy_cfg(EXACT_08_BASELINE_RERANK),
            "metrics": exact_metrics,
        }
        all_candidates.append(
            make_candidate(
                bundle_results["exact_08_baseline_rerank"]["label"],
                bundle_name,
                "exact_08_baseline_rerank",
                exact_metrics,
                {"rerank": copy_cfg(EXACT_08_BASELINE_RERANK)},
            )
        )
        if exceeded:
            bundle_results["rerank_guard_triggered"] = True

        if not bundle_results["rerank_guard_triggered"]:
            for cfg in RERANK_CONFIGS:
                metrics, exceeded = evaluate_rerank_variant(base_state, q_pids, g_pids, q_camids, g_camids, cfg, f"{bundle_name} rerank")
                record = {
                    "label": f"{bundle_name} rerank {cfg['label']}",
                    "config": copy_cfg(cfg),
                    "metrics": metrics,
                }
                bundle_results["rerank_sweep"].append(record)
                all_candidates.append(make_candidate(record["label"], bundle_name, "rerank", metrics, {"rerank": copy_cfg(cfg)}))
                if exceeded:
                    bundle_results["rerank_guard_triggered"] = True
                    break

        aqe_feature_cache = {}
        for k in AQE_K_VALUES:
            expanded = average_query_expansion(all_features, k=k, iterations=1)
            aqe_feature_cache[k] = expanded
            q_aqe = expanded[:len(q_pids)]
            g_aqe = expanded[len(q_pids):]
            started = time.time()
            metrics = evaluate_cosine_features(q_aqe, g_aqe, q_pids, g_pids, q_camids, g_camids)
            elapsed = time.time() - started
            record = {
                "label": f"{bundle_name} AQE k={k}",
                "config": {"k": int(k), "iterations": 1},
                "metrics": {**metrics, "duration_sec": float(elapsed)},
            }
            bundle_results["aqe_sweep"].append(record)
            print_metrics(record["label"], record["metrics"], duration_sec=elapsed)
            all_candidates.append(make_candidate(record["label"], bundle_name, "aqe", record["metrics"], {"aqe": record["config"]}))

        expanded_iter2 = average_query_expansion(all_features, k=AQE_ITER2_K, iterations=2)
        q_iter2 = expanded_iter2[:len(q_pids)]
        g_iter2 = expanded_iter2[len(q_pids):]
        started = time.time()
        iter2_metrics = evaluate_cosine_features(q_iter2, g_iter2, q_pids, g_pids, q_camids, g_camids)
        elapsed = time.time() - started
        bundle_results["aqe_iter2"] = {
            "label": f"{bundle_name} AQE iter2 k={AQE_ITER2_K}",
            "config": {"k": int(AQE_ITER2_K), "iterations": 2},
            "metrics": {**iter2_metrics, "duration_sec": float(elapsed)},
        }
        print_metrics(bundle_results["aqe_iter2"]["label"], bundle_results["aqe_iter2"]["metrics"], duration_sec=elapsed)
        all_candidates.append(
            make_candidate(
                bundle_results["aqe_iter2"]["label"],
                bundle_name,
                "aqe_iter2",
                bundle_results["aqe_iter2"]["metrics"],
                {"aqe": bundle_results["aqe_iter2"]["config"]},
            )
        )

        def rerank_key(cfg):
            return (int(cfg["k1"]), int(cfg["k2"]), float(cfg["lambda_value"]), cfg["label"])

        cross_aqe_values = sorted({int(k) for k in CROSS_AQE_K_VALUES if int(k) in aqe_feature_cache})
        cross_configs_by_k = {k: [copy_cfg(cfg) for cfg in RERANK_CONFIGS] for k in cross_aqe_values}
        for pair in GUARANTEED_AQE_RERANK_PAIRS:
            aqe_k = int(pair["aqe_k"])
            if aqe_k not in aqe_feature_cache:
                continue
            configs = cross_configs_by_k.setdefault(aqe_k, [])
            guaranteed_cfg = copy_cfg(pair["rerank"])
            if rerank_key(guaranteed_cfg) not in {rerank_key(cfg) for cfg in configs}:
                configs.append(guaranteed_cfg)

        if cross_configs_by_k and not bundle_results["rerank_guard_triggered"]:
            cross_guard_triggered = False
            for k in sorted(cross_configs_by_k):
                expanded = aqe_feature_cache[k]
                cross_configs = cross_configs_by_k[k]
                max_cross_k1 = max(cfg["k1"] for cfg in cross_configs)
                print(f"  Building rerank state for {bundle_name} AQE k={k} cross grid ...")
                aqe_state = build_rerank_state(expanded, max_k1=max_cross_k1)
                for cfg in cross_configs:
                    metrics, exceeded = evaluate_rerank_variant(aqe_state, q_pids, g_pids, q_camids, g_camids, cfg, f"{bundle_name} AQE(k={k})+rerank")
                    record = {
                        "label": f"{bundle_name} AQE(k={k}) + {cfg['label']}",
                        "aqe": {"k": int(k), "iterations": 1},
                        "rerank": copy_cfg(cfg),
                        "metrics": metrics,
                    }
                    bundle_results["aqe_rerank_cross"].append(record)
                    all_candidates.append(make_candidate(record["label"], bundle_name, "aqe_rerank_cross", metrics, {"aqe": record["aqe"], "rerank": record["rerank"]}))
                    if exceeded:
                        bundle_results["rerank_guard_triggered"] = True
                        cross_guard_triggered = True
                        break
                del aqe_state
                gc.collect()
                if DEVICE.startswith("cuda"):
                    torch.cuda.empty_cache()
                if cross_guard_triggered:
                    break

        results["feature_sets"][bundle_name] = bundle_results
        del base_state
        gc.collect()
        if DEVICE.startswith("cuda"):
            torch.cuda.empty_cache()

    ranked_all = sorted(all_candidates, key=lambda candidate: metric_sort_key(candidate["metrics"]), reverse=True)
    results["top_candidates"] = ranked_all[:10]
    results["best_overall"] = ranked_all[0] if ranked_all else None
    results["metadata"]["total_runtime_sec"] = float(time.time() - total_start)

    print("\nTop overall candidates:")
    for candidate in results["top_candidates"]:
        print_metrics(candidate["label"], candidate["metrics"])

    del model
    for bundle in feature_bundles.values():
        del bundle
    gc.collect()
    if DEVICE.startswith("cuda"):
        torch.cuda.empty_cache()
    return results

In [ ]:
# Primary eval: VeRi-776-trained checkpoint
results_veri = run_full_eval(
    WEIGHTS_VERI,
    "Checkpoint A: vehicle_transreid_vit_base_veri776.pth (VeRi-776 trained, 256x256 + SIE v7 sweep)",
)

In [ ]:
import json

summary = {
    "img_size": list(IMG_SIZE),
    "ten_crop_resize": list(TEN_CROP_RESIZE),
    "vit_model": "vit_base_patch16_clip_224.openai",
    "aqe_k_values": list(AQE_K_VALUES),
    "aqe_iter2_k": AQE_ITER2_K,
    "max_time_per_rerank_sec": MAX_TIME_PER_RERANK,
    "rerank_configs": [copy_cfg(cfg) for cfg in RERANK_CONFIGS],
    "exact_08_baseline_rerank": copy_cfg(EXACT_08_BASELINE_RERANK),
    "eval_no_sie": EVAL_NO_SIE,
    "checkpoints": {
        Path(WEIGHTS_VERI).name: results_veri,
    },
    "best_overall": results_veri.get("best_overall"),
    "top_candidates": results_veri.get("top_candidates", []),
}

with open(RESULTS_PATH, "w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2)

print(f"Wrote {RESULTS_PATH}")
print(json.dumps(summary, indent=2))